# Binary Liver Segmentation

## Setup

In [27]:
# Installation of torch and torchvision compatible to Kaggle
!pip install torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121 -q
!pip install nibabel huggingface_hub[hf_transfer]==0.24.0 -q

In [28]:
# Import libraries
import os
import random
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from PIL import Image
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [ ]:
# For reproducibility
seed = 9
random.seed(seed)
np.random.seed(seed)
num_workers = 0 # For DataLoader

In [ ]:
# Rechecking if setup with Kaggle works or not
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
print(torch.__version__)

Downloading directly from HuggingFace into Kaggle local directory
- Created a token in HuggingFace and used the token to download required files into Kaggle 
- Needed only 2 files hence built a function that does it rather than downloading all files

In [ ]:
from huggingface_hub import hf_hub_download
from kaggle_secrets import UserSecretsClient

# Credential configuration omitted

In [ ]:
num_cases = 75 # Downloaded 75 cases
case_names = [f"BDMAP_{i:08d}" for i in range(1, num_cases + 1)]

local_dir = "/kaggle/working/data" 
os.makedirs(local_dir, exist_ok=True) # Creating a new directory

files = ["ct.nii.gz", "segmentations/liver.nii.gz"] # List of needed files
repo_id = "AbdomenAtlas/AbdomenAtlas1.0Mini" # Repo ID from HF

for case_name in case_names:
    case_dir = os.path.join(local_dir, case_name) # Loop through each files to create two files
    seg_dir = os.path.join(case_dir, "segmentations") # One for Full volume and other for Liver volume
    os.makedirs(seg_dir, exist_ok=True)

    for filename in files:
        local_path = os.path.join(local_dir, case_name, filename)
        if not os.path.exists(local_path):
            hf_hub_download(
                repo_id=repo_id,
                repo_type="dataset",
                filename=f"{case_name}/{filename}",
                local_dir=local_dir,
                token=hf_token
            )

## Preprocessing

* Rather than using the whole volume we use slices
* Kept images with HU between -100 to 400 as anything below or above that don't contain information for us
* To reduce compute while keeping the dataset balanced, we randomly keep only about 20% of slices that contain no liver and discard the rest
* Windowing: Keeping anything below minimum value to minimum value & anything above maximum value to maximum value
* Normalization: Values between 0 and 1
* Bilinear interpolation: Preserves intensity smoothness for CT
* Used NumPy for storage of information, as it is memory efficient

In [ ]:
def preprocess(case_names):
    min_val = -100
    max_val = 400 
    
    save_dir = "/kaggle/working/slices"

    os.makedirs(f"{save_dir}/ct", exist_ok=True)
    os.makedirs(f"{save_dir}/mask", exist_ok=True)

    index = []

    for case in case_names:
        ct = nib.load(f"/kaggle/working/data/{case}/ct.nii.gz").get_fdata()
        mask = nib.load(f"/kaggle/working/data/{case}/segmentations/liver.nii.gz").get_fdata()

        for s in range(ct.shape[2]):
            if mask[:, :, s].sum() == 0 and np.random.rand() > 0.2: # Balancing
                continue

            ct_path = f"{save_dir}/ct/{case}_slice{s}.npy"
            mask_path = f"{save_dir}/mask/{case}_slice{s}.npy"

            if not os.path.exists(ct_path):
                ct_slice = np.clip(ct[:, :, s], min_val, max_val) # Windowing
                ct_slice = (ct_slice - min_val) / (max_val - min_val) # Normalization

                ct_slice = np.array(Image.fromarray(ct_slice.astype(np.float32)).resize((256, 256), resample=Image.BILINEAR)) # Resize CT
                mask_slice = np.array(Image.fromarray(mask[:, :, s].astype(np.uint8)).resize((256, 256), resample=Image.NEAREST)) # Resize Mask
                mask_slice = (mask_slice > 0).astype(np.uint8) # Convert mask to binary format

                np.save(ct_path, ct_slice.astype(np.float32))
                np.save(mask_path, mask_slice)

            index.append((ct_path, mask_path))

    return index

### Train, Validation and Test split

- Split by cases not slices.
- All slices from one patient go to the same split.
- Splitting by slices would cause data leakage since consecutive slices from the same patient are nearly identical.

In [34]:
train_cases, test_cases = train_test_split(case_names, test_size=0.1, random_state=seed)
train_cases, val_cases  = train_test_split(train_cases, test_size=0.1, random_state=seed)

print(f"Number of train cases: {len(train_cases)}")
print(f"Number of valdiation cases: {len(val_cases)}")
print(f"Number of test cases: {len(test_cases)}")

* Test: 0.2N
* Validation: 0.1×(1−0.2)N=0.08N
* Train: (1−0.2−0.08)N=0.72N

In [ ]:
# Preprocess the indexes
train_index = preprocess(train_cases)
val_index   = preprocess(val_cases)
test_index  = preprocess(test_cases)

print(f"Number of train index: {len(train_index)}")
print(f"Number of valdiation index: {len(val_index)}")
print(f"Number of test index: {len(test_index)}")

In [ ]:
# Custom Dataset creation
class LiverDataset(Dataset):
    def __init__(self, index):
        self.index = index

    def __len__(self):
        return len(self.index)

    def __getitem__(self, idx):
        ct_path, mask_path = self.index[idx]
        ct = torch.tensor(np.load(ct_path), dtype=torch.float32).unsqueeze(0) # Loads CT & converts them to PyTorch tensors
        mask = torch.tensor((np.load(mask_path) > 0.5).astype(np.float32)).unsqueeze(0) # unsqueeze(0) adds channel dimension
        return ct, mask

In [ ]:
train_dataset = LiverDataset(train_index)
val_dataset   = LiverDataset(val_index)
test_dataset  = LiverDataset(test_index)

train_loader = DataLoader(train_dataset,batch_size=8,shuffle=True,num_workers=num_workers) # shuffle=True prevents the model from memorizing the order of samples
val_loader   = DataLoader(val_dataset,batch_size=8,shuffle=False,num_workers=num_workers)
test_loader  = DataLoader(test_dataset,batch_size=8,shuffle=False,num_workers=num_workers)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Test batches:  {len(test_loader)}")

DoubleConv: Two convolution blocks to extract robust spatial features  
Encoder: Capture high-level context by reducing spatial size and increasing channels  
Decoder: Recover spatial details and improve localization

Bottleneck: Captures the most abstract features before reconstruction  

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.conv = nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
        nn.Dropout2d(0.2),
        nn.Conv2d(out_ch, out_ch, 3, padding=1),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.conv(x)

class UNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, features=[32, 64, 128, 256]):
        super().__init__()
        self.encoders = nn.ModuleList()
        self.pool = nn.MaxPool2d(2, 2)
        self.upconvs = nn.ModuleList()
        self.decoders = nn.ModuleList()
        for f in features:
            self.encoders.append(DoubleConv(in_ch, f))
            in_ch = f
        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        for f in reversed(features):
            self.upconvs.append(nn.ConvTranspose2d(f*2, f, 2, stride=2))
            self.decoders.append(DoubleConv(f*2, f))
        self.final = nn.Conv2d(features[0], out_ch, 1)

    def forward(self, x):
        skips = []
        for enc in self.encoders:
            x = enc(x)
            skips.append(x)
            x = self.pool(x)
        x = self.bottleneck(x)
        skips = skips[::-1]
        for i in range(len(self.decoders)):
            x = self.upconvs[i](x)
            if x.shape != skips[i].shape:
                x = torch.nn.functional.interpolate(x, size=skips[i].shape[2:])
            x = torch.cat([skips[i], x], dim=1)
            x = self.decoders[i](x)
        return torch.sigmoid(self.final(x))

model = UNet().to(device)

1. Dice Loss = Measures overlap between prediction and target
2. Dice Score = Evaluation metric after thresholding
3. Combined Loss = Combines BCE with Dice

### Evaluation metrics

In [39]:
def dice_loss(pred, target, smooth=1e-6):
    intersection = (pred * target).sum(dim=(2, 3))
    return 1 - ((2*intersection + smooth) / (pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3)) + smooth)).mean()

def dice_score(pred, target, smooth=1e-6):
    pred_b = (pred > 0.5).float()
    intersection = (pred_b * target).sum(dim=(2, 3))
    return ((2*intersection + smooth) / (pred_b.sum(dim=(2, 3)) + target.sum(dim=(2, 3)) + smooth)).mean().item()

def combined_loss(pred, target):
    return nn.BCELoss()(pred, target) + dice_loss(pred, target)

In [ ]:
# Based on Karpathy's guide from website
def overfit_one_batch(model, loader, device, steps=50):
    opt = torch.optim.Adam(model.parameters(), lr=3e-4)
    images, masks = next(iter(loader))
    images, masks = images.to(device), masks.to(device)
    for step in range(steps):
        opt.zero_grad()
        preds = model(images)
        loss  = combined_loss(preds, masks)
        loss.backward()
        opt.step()
        if step % 10 == 0:
            print(f"Step {step:3d} - Loss: {loss.item():.4f} - Dice: {dice_score(preds, masks):.4f}")

model = UNet().to(device)
overfit_one_batch(model, train_loader, device)

### Model Training

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss, total_dice = 0, 0
    for images, masks in loader:
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        preds = model(images)
        loss  = combined_loss(preds, masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        total_dice += dice_score(preds, masks)
    return total_loss / len(loader), total_dice / len(loader)

def validate(model, loader, device):
    model.eval()
    total_loss, total_dice = 0, 0
    with torch.no_grad():
        for images, masks in loader:
            images, masks = images.to(device), masks.to(device)
            preds = model(images)
            total_loss += combined_loss(preds, masks).item()
            total_dice += dice_score(preds, masks)
    return total_loss / len(loader), total_dice / len(loader)

model     = UNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)

NUM_EPOCHS    = 30
best_val_dice = 0
train_losses, val_losses, train_dices, val_dices = [], [], [], []

for epoch in range(NUM_EPOCHS):
    train_loss, train_dice = train_one_epoch(model, train_loader, optimizer, device)
    val_loss,val_dice = validate(model, val_loader, device)
    scheduler.step(val_loss)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_dices.append(train_dice)
    val_dices.append(val_dice)

    if val_dice > best_val_dice:
        best_val_dice = val_dice
        torch.save(model.state_dict(), "modelbest_v4.pth")

    print(f"Epoch{epoch+1:2d} - Train Loss: {train_loss:.4f} - Val Loss: {val_loss:.4f} - Train Dice: {train_dice:.4f} - Val Dice: {val_dice:.4f}")

print(f"Best Val Dice: {best_val_dice:.4f}")

Best validation Dice score of 0.8962

### Visualization

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(train_losses,label="Train")
ax1.plot(val_losses, label="Val")
ax1.set_title("Loss / Epoch")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()

ax2.plot(train_dices, label="Train")
ax2.plot(val_dices, label="Val")
ax2.set_title("Dice Score / Epoch")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Dice")
ax2.legend()

plt.tight_layout()
plt.show()

Shows slight overfitting

#### Ground Truth vs Predicted Mask

In [ ]:
model.load_state_dict(torch.load("modelbest_v4.pth", weights_only=True)) # copies each weight into the correct layer of the model
model.eval() 

liver_indices = [i for i in range(len(test_dataset)) if test_dataset[i][1].sum() > 0] # slices that have liver in GT
fig, axes = plt.subplots(5, 3, figsize=(12, 20))
indices = np.random.choice(liver_indices, 5, replace=False)

with torch.no_grad():
    for row, idx in enumerate(indices):
        ct, mask = test_dataset[idx]
        pred = model(ct.unsqueeze(0).to(device))

        axes[row, 0].imshow(ct.squeeze(),cmap="gray")
        axes[row, 0].set_title("CT Slice")
        axes[row, 0].axis("off")

        axes[row, 1].imshow(mask.squeeze(),cmap="gray")
        axes[row, 1].set_title("Ground Truth")
        axes[row, 1].axis("off")

        axes[row, 2].imshow(pred.squeeze().cpu() > 0.5, cmap="gray")
        axes[row, 2].set_title("Prediction")
        axes[row, 2].axis("off")

plt.tight_layout()
plt.show()

Reference:
1. Ronneberger, O., Fischer, P., & Brox, T. (2015). U-Net: Convolutional Networks for Biomedical Image Segmentation. MICCAI 2015
https://arxiv.org/abs/1505.04597
2. https://huggingface.co/datasets/AbdomenAtlas/AbdomenAtlas1.0Mini
3. Sudre, C. et al. (2017). Generalised Dice Overlap as a Deep Learning Loss Function for Highly Unbalanced Segmentations. MICCAI 2017
https://arxiv.org/abs/1707.03237
4. Karpathy, A. (2019). A Recipe for Training Neural Networks. https://karpathy.github.io/2019/04/25/recipe/

AI tools employed: Claude for Directory management, idea about downloading dataset, finding relevent papers, explaination of concepts and outputs
HuggingFace for idea about download of data into Kaggle using HF_token